# Bharat-Tiny-LLM v2: QLoRA Fine-tuning

**Goal**: Fine-tune Qwen2.5-1.5B (with 300 new Devanagari tokens) using QLoRA.
This teaches the model to USE the new tokens for coherent Hindi generation.

**Hardware**: T4 GPU (Colab free). ~1 hour for 20K chunks, 2 epochs.

**DO NOT run cells manually** — use Runtime → Run all.

In [ ]:
!pip install -q transformers torch datasets accelerate huggingface_hub peft bitsandbytes

import json, os, math, time, random, requests, gzip, shutil
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from huggingface_hub import login, HfApi
from google.colab import drive
from datasets import load_dataset

# === CONFIG ===
HF_TOKEN = "hf_YOUR_TOKEN_HERE"
MODEL_ID = 'eulogik/Bharat-Tiny-LLM-v2'

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/brahmi_lora'
os.makedirs(DRIVE_DIR, exist_ok=True)

login(token=HF_TOKEN)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Drive: {DRIVE_DIR}')

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
)

model = prepare_model_for_kbit_training(model)

new_ids = sorted(
    tid for tid, t in tok.added_tokens_decoder.items()
    if not str(t).startswith('<') and not getattr(t, 'special', False)
)
print(f'Found {len(new_ids)} new tokens')
print(f'Params: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
DATA_URL = 'https://huggingface.co/eulogik/Bharat-Tiny-LLM-v2/resolve/main/train_gold_v3.jsonl.gz'
DATA_FILE = '/content/train_gold_v3.jsonl'

if not os.path.exists(DATA_FILE):
    print('Downloading...')
    r = requests.get(DATA_URL, stream=True,
                     headers={'Authorization': f'Bearer {HF_TOKEN}'})
    r.raise_for_status()
    with open('/content/data.gz', 'wb') as f:
        shutil.copyfileobj(r.raw, f)
    with gzip.open('/content/data.gz', 'rb') as gz, open(DATA_FILE, 'wb') as f:
        shutil.copyfileobj(gz, f)
    os.remove('/content/data.gz')

dataset = load_dataset('text', data_files=DATA_FILE, split='train')
print(f'Loaded {len(dataset)} rows')

In [ ]:
CHUNKS_FILE = os.path.join(DRIVE_DIR, 'lora_chunks.json')

if os.path.exists(CHUNKS_FILE):
    with open(CHUNKS_FILE) as f:
        all_chunks = json.load(f)
    print(f'Loaded {len(all_chunks)} chunks from Drive')
else:
    all_chunks = []
    for i, example in enumerate(dataset):
        text = example.get('text', '')
        if isinstance(text, str) and text.startswith('{'):
            try:
                data = json.loads(text)
                text = ' '.join(m['content'] for m in data.get('messages', []))
            except:
                pass
        ids = tok.encode(text)
        for j in range(0, len(ids), 512):
            chunk = ids[j:j+512]
            if len(chunk) >= 10:
                all_chunks.append(chunk)
        if i % 50000 == 0 and i > 0:
            print(f'  Processed {i}/{len(dataset)} rows...')

    random.shuffle(all_chunks)
    if len(all_chunks) > 20000:
        all_chunks = all_chunks[:10000]
    with open(CHUNKS_FILE, 'w') as f:
        json.dump(all_chunks, f)
    print(f'Total chunks: {len(all_chunks)} (saved to Drive)')

split = int(len(all_chunks) * 0.95)
train_chunks = all_chunks[:split]
val_chunks = all_chunks[split:]
print(f'Train: {len(train_chunks)}, Val: {len(val_chunks)}')

In [ ]:
class ChunkDataset(Dataset):
    def __init__(self, chunks, max_len=512):
        self.chunks = chunks
        self.max_len = max_len

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        ids = self.chunks[idx][:self.max_len]
        padded = ids + [tok.pad_token_id] * (self.max_len - len(ids))
        mask = [1] * len(ids) + [0] * (self.max_len - len(ids))
        return {
            'input_ids': torch.tensor(padded, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(padded, dtype=torch.long),
        }

train_dataset = ChunkDataset(train_chunks)
val_dataset = ChunkDataset(val_chunks)
print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

In [ ]:
CKPT_DIR = os.path.join(DRIVE_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='paged_adamw_8bit',
    max_grad_norm=1.0,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Resume from last checkpoint if exists
checkpoints = [d for d in os.listdir(CKPT_DIR) if d.startswith('checkpoint-')]
resume = None
if checkpoints:
    latest = max(checkpoints, key=lambda x: int(x.split('-')[1]))
    resume = os.path.join(CKPT_DIR, latest)
    print(f'Resuming from {latest}')

stats = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
print(f'Effective batch size: {stats}')
print('Starting training...')
trainer.train(resume_from_checkpoint=resume)

In [ ]:
# Save LoRA adapter to Drive and HF
ADAPTER_DIR = os.path.join(DRIVE_DIR, 'lora_adapter')
model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
print(f'LoRA saved to {ADAPTER_DIR}')

api = HfApi(token=HF_TOKEN)
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id='eulogik/Bharat-Tiny-LLM-v2-LoRA',
    repo_type='model',
    commit_message='LoRA: 2 epochs, rank=16, 2e-4',
)
print('Uploaded LoRA to HF!')

In [ ]:
# Merge LoRA into base and upload
print('Merging LoRA into base model...')
merged = model.merge_and_unload()

MERGED_DIR = os.path.join(DRIVE_DIR, 'merged')
merged.save_pretrained(MERGED_DIR)
tok.save_pretrained(MERGED_DIR)
print(f'Merged model saved to {MERGED_DIR}')

api = HfApi(token=HF_TOKEN)
api.upload_folder(
    folder_path=MERGED_DIR,
    repo_id='eulogik/Bharat-Tiny-LLM-v2',
    repo_type='model',
    commit_message='LoRA merged: rank=16, 2 epochs',
)
print('Merged model uploaded to HF!')

In [ ]:
prompts = [
    'मैं आपको बताना चाहता हूँ कि',
    'भारत की राजधानी',
    'नमस्ते, आप कैसे हैं? मैं',
]

for p in prompts:
    inputs = tok(p, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=60,
        temperature=0.3,
        top_p=0.85,
        repetition_penalty=1.25,
        do_sample=True,
    )
    gen = tok.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    print(f'Prompt: {p}')
    print(f'  -> {gen}\n')